In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [2]:
pip install langchain openai langchain-openai langchain-experimental langchainhub

Defaulting to user installation because normal site-packages is not writeable

   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2/3 [langchain-experimental]
   -------------------------- ------------- 2

### 5-2 Agents

### Agents 사용 예

In [3]:
import langchain
langchain.verbose = True

In [7]:
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent, load_tools
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)
tools = load_tools(["terminal"], allow_dangerous_tools=True)

prompt = hub.pull("hwchase17/react")

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result = agent_executor.invoke({"input":"sample_data 디렉터리에 있는 파일 목록을 알려줘, 터미널을 사용할 때 인코딩도 알아서 맞춰줘."})



> Entering new AgentExecutor chain...
Thought: sample_data 디렉터리에 있는 파일 목록을 확인하기 위해 터미널 명령어를 실행해야 한다. Windows 환경이므로 인코딩 문제를 고려하여 명령어를 실행하고 결과를 확인하겠다.
Action: terminal
Action Input: chcp 65001Executing command:
 chcp 65001
Active code page: 65001


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


Thought: 코드 페이지를 UTF-8(65001)로 변경했으니, 이제 sample_data 디렉터리의 파일 목록을 확인하는 명령어를 실행하겠다.
Action: terminal
Action Input: dir sample_dataExecuting command:
 dir sample_data
 Volume in drive C has no label.
 Volume Serial Number is 6C6F-4BFC

 Directory of c:\강의 교안\LLM

File Not Found
Thought: sample_data 디렉터리가 현재 경로(c:\강의 교안\LLM) 내에 없거나 이름이 다를 수 있다. 현재 경로를 확인하고, sample_data 디렉터리가 존재하는지 확인한 후, 다른 위치에 있을 가능성도 고려해야 한다.
Action: terminal
Executing command:
 cd
c:\강의 교안\LLM
Thought: 현재 경로는 c:\강의 교안\LLM 이고, 이 경로 내에 sample_data 디렉터리가 없었다. 상위 디렉터리나 다른 위치에 sample_data가 있을 수 있으니, 상위 디렉터리로 이동하여 다시 확인하거나 sample_data 폴더를 검색해보겠다.
Action: terminal
Executing command:
 cd ..
Thought: 상위 디렉터리로 이동했으니, 다시 sample_data 디렉터리가 있는지 확인하겠다.
Action: terminal
Action Input: dir sample_dataExecuting command:
 dir sample_data
 Volume in drive C has no label.
 Volume Serial Number is 6C6F-4BFC

 Directory of c:\강의 교안\LLM

File Not Found
Thought: 상위 디렉터리로 이동했지만 여전히 sample_data 디렉터리가 없다고 나온다. 현재 위치가 c:\강의 교안 이고, 이 위치에서 sample_da

In [8]:
print(result["output"])

sample_data 디렉터리(C:\Users\User\sample_data) 내 파일 목록은 다음과 같습니다.

- 02. folium_US_Unemployment_Oct2012.csv
- asil_school_data.csv
- cctv_result.csv
- seoul_school_ranking_2024.csv
- seoul_school_stat.csv
- seoul_school_stats.csv
- seoul_school_stats_gusplit.csv
- seoul_school_stats_with_go_mean.csv
- sport_test.csv


## Tools

In [9]:
from langchain_classic.tools import Tool

def my_super_func(param):
    return "42"

tools = [
    Tool.from_function(
        func=my_super_func,
        name="The_Answer",
        description="생명, 우주, 그리고 모든 것에 대한 궁극적인 질문의 답"
    ),
]

In [10]:
agent = create_react_agent(llm, tools,prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose = True)
result = agent_executor.invoke({"input": "이 세계의 진리를 알려주세요"})
print(result)



> Entering new AgentExecutor chain...
Question: 이 세계의 진리를 알려주세요
Thought: 이 세계의 진리에 대해 묻는 질문이므로, '생명, 우주, 그리고 모든 것에 대한 궁극적인 질문의 답'을 제공하는 도구를 사용해 답을 찾아야겠다.
Action: The_Answer
Action Input: 이 세계의 진리42Thought: The observation "42" is a reference to the famous answer to the ultimate question of life, the universe, and everything from Douglas Adams' "The Hitchhiker's Guide to the Galaxy." This suggests that the ultimate truth or answer to the question about the world's truth is symbolically "42."
Final Answer: 이 세계의 진리에 대한 궁극적인 답은 "42"입니다.

> Finished chain.
{'input': '이 세계의 진리를 알려주세요', 'output': '이 세계의 진리에 대한 궁극적인 답은 "42"입니다.'}


In [12]:
from langchain_openai import ChatOpenAI
from langchain_classic.prompts import PromptTemplate
from langchain_classic import LLMChain

summarize_template = """아래의 글을 결론만 한 마디로 요약해 주세요.
{input}
"""
summarize_prompt = PromptTemplate(
    input_variables=["input"],
    template=summarize_template,
)

chat = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)
summarize_chain = LLMChain(llm=chat, prompt=summarize_prompt)

tools = [ 
    Tool.from_function(
        func=summarize_chain.run,
        name="Summarizer",
        description="Text summarizer"
    )
]


C:\Users\User\AppData\Local\Temp\ipykernel_9692\3455085236.py:14: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  summarize_chain = LLMChain(llm=chat, prompt=summarize_prompt)


In [13]:
agnet = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

text = """다음을 요약해 주세요.
안녕하세요! 저는 ChatGPT라고 불리는 AI 언어 모델입니다.
OpenAI가 개발한 GPT-3.5 아키텍처를 기반으로 합니다.
저는 자연어 이해와 생성을 전문으로 하며, 다양한 주제에 대한 질문에 답하거나, 대화를 나누는 것을 잘합니다.
제 트레이닝 데이터는 2021년 9월까지의 정보를 기반으로 하기 때문에, 그 이후의 사건에 대해서는 지식이 없습니다.
하지만, 가능한 한 도움을 드리기 위해 노력할 것입니다.
질문이나 대화, 정보 공유 등, 어떤 도움이든 편하게 말씀해 주세요! 잘 부탁드립니다."""

result = agent_executor.invoke({"input":text})
print(result)




> Entering new AgentExecutor chain...
Question: 다음을 요약해 주세요.
안녕하세요! 저는 ChatGPT라고 불리는 AI 언어 모델입니다.
OpenAI가 개발한 GPT-3.5 아키텍처를 기반으로 합니다.
저는 자연어 이해와 생성을 전문으로 하며, 다양한 주제에 대한 질문에 답하거나, 대화를 나누는 것을 잘합니다.
제 트레이닝 데이터는 2021년 9월까지의 정보를 기반으로 하기 때문에, 그 이후의 사건에 대해서는 지식이 없습니다.
하지만, 가능한 한 도움을 드리기 위해 노력할 것입니다.
질문이나 대화, 정보 공유 등, 어떤 도움이든 편하게 말씀해 주세요! 잘 부탁드립니다.
Thought: 이 문장의 내용을 간결하게 요약하면, ChatGPT가 무엇인지, 어떤 기능을 갖고 있는지, 그리고 제한 사항과 도움 의지를 설명하고 있다.
Final Answer: 저는 OpenAI가 개발한 GPT-3.5 기반 AI 언어 모델 ChatGPT로, 자연어 이해와 생성에 능하며 2021년 9월까지의 정보를 바탕으로 다양한 질문에 답하고 대화를 나눕니다. 이후 정보는 알지 못하지만, 최대한 도움을 드리기 위해 노력합니다.

> Finished chain.
{'input': '다음을 요약해 주세요.\n안녕하세요! 저는 ChatGPT라고 불리는 AI 언어 모델입니다.\nOpenAI가 개발한 GPT-3.5 아키텍처를 기반으로 합니다.\n저는 자연어 이해와 생성을 전문으로 하며, 다양한 주제에 대한 질문에 답하거나, 대화를 나누는 것을 잘합니다.\n제 트레이닝 데이터는 2021년 9월까지의 정보를 기반으로 하기 때문에, 그 이후의 사건에 대해서는 지식이 없습니다.\n하지만, 가능한 한 도움을 드리기 위해 노력할 것입니다.\n질문이나 대화, 정보 공유 등, 어떤 도움이든 편하게 말씀해 주세요! 잘 부탁드립니다.', 'output': '저는 OpenAI가 개발한 GPT-3.5 기반 AI 언어 모델 ChatGPT로, 자연어 이해와 생성에 능하며 2021년 9

### Function calling을 사용하는 OpenAI Funcitons Agent

In [14]:
from langchain_classic.agents import load_tools
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
tools = load_tools (["terminal"], allow_dangerous_tools=True)
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input":"sample_data디렉터리에 있는 파일 목록을 알려줘, 터미널을 사용할 때 인코딩도 알아서 맞춰서 해줘."})



> Entering new AgentExecutor chain...

Invoking: `terminal` with `{'commands': ['chcp 65001', 'dir sample_data']}`


Executing command:
 ['chcp 65001', 'dir sample_data']
Parameter format not correct - 65001;dir


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_community\tools\shell\tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(



Invoking: `terminal` with `{'commands': ['chcp 65001', 'dir sample_data']}`


Executing command:
 ['chcp 65001', 'dir sample_data']
Parameter format not correct - 65001;dir

Invoking: `terminal` with `{'commands': ['chcp 65001', 'dir sample_data']}`


Executing command:
 ['chcp 65001', 'dir sample_data']
Parameter format not correct - 65001;dir

Invoking: `terminal` with `{'commands': ['chcp 65001', '&&', 'dir sample_data']}`


Executing command:
 ['chcp 65001', '&&', 'dir sample_data']
Parameter format not correct - 65001;

Invoking: `terminal` with `{'commands': ['chcp 65001', 'dir sample_data']}`


Executing command:
 ['chcp 65001', 'dir sample_data']
Parameter format not correct - 65001;dir

Invoking: `terminal` with `{'commands': ['chcp 65001', '&', 'dir sample_data']}`


Executing command:
 ['chcp 65001', '&', 'dir sample_data']
Parameter format not correct - 65001;
 Volume in drive C has no label.
 Volume Serial Number is 6C6F-4BFC

 Directory of c:\강의 교안\LLM

File Not Found
sa

{'input': 'sample_data디렉터리에 있는 파일 목록을 알려줘, 터미널을 사용할 때 인코딩도 알아서 맞춰서 해줘.',
 'output': 'sample_data 디렉터리에 파일이 없습니다. 다른 작업을 도와드릴까요?'}

### 여러 도구를 한 번에 사용

In [23]:
pip install ddgs

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/5 [socksio]
   ---------------- ----------------------- 2/5 [hpack]
   ---------------- ----------------------- 2/5 [hpack]
   ------------------------ --------------- 3/5 [h2]
   ------------------------ --------------- 3/5 [h2]
   -------------------------------- ------- 4/5 [ddgs]
   -------------------------------- ------- 4/5 [ddgs]
   -------------------------------- ------- 4/5 [ddgs]
   ---------------------------------------- 5/5 [ddgs]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [24]:
import langchain

langchain.debug = True
langchain.verbose = False

In [25]:
from langchain_classic.agents import load_tools
from langchain_classic.agents import AgentExecutor, create_openai_functions_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
tools = load_tools(["ddg-search"])
prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_functions_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools = tools, verbose=True)

result = agent_executor.invoke({"input":"서울과 부산의 날씨를 알려줘"})



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': '서울 날씨'}`


We would like to show you a description here but the site won’t allow us. Hourly weather forecast in 서울특별시, 서울시, 대한민국. Check current conditions in 서울특별시, 서울시, 대한민국 with radar, hourly, and more. Weather Channel 및 Weather.com의 가장 정확한 최고온, 최저온, 강수 확률 정보가 제공되는 서울 특별시, 대한민국 10일 일기예보로 준비하세요 1 day ago · 사용자 친화적인 그래픽 인터페이스를 통해 서울특별시 의 풍속 및 풍향, 기압, 기온, 습도, 가시거리 등의 정보가 제공됩니다. 또한, 해당 일자의 해와 달의 일출 및 일몰 시간도 함께 안내됩니다. MSN 날씨과 (와) 함께 서울의 오늘, 오늘 밤, 내일에 대한 정확한 시간별 예보와 함께 10일간의 일일 예보 및 기상 레이더를 확인하세요. 강수량 ...
Invoking: `duckduckgo_search` with `{'query': '부산 날씨'}`


2025. 11. 10. · 내일 · 시각: 01시 · 날씨: 구름 많음 · 기온(체감온도) -4℃(-10℃) · 체감온도: -10℃ · 강수량: - · - · 강수확률: 20% · 바람: 북서풍 약간 강 4m/s ... 2025. 5. 9. · 부산지방기상청만의 날씨 서비스를 소개합니다ㅣ부산지방기상청 사무실&장비 공개! 11K views · 8 months ago. #기상청 #국민정책기자단 #부산지방기상청 ...more ... 2025. 12. 3. · 부산 시간별날씨 ; 04시 · 4.5m/s · 습도 20% ; 05시 · 5.3m/s · 습도 20% ; 06시 · 5.7m/s · 습도 25% ; 07시 · 5

In [26]:
print(result["output"])

현재 서울과 부산의 날씨 정보를 알려드리겠습니다.

서울의 현재 날씨 정보는 정확한 데이터가 제공되지 않았으나, 일반적으로 서울의 기온, 습도, 풍속 등의 정보를 확인할 수 있습니다.

부산의 현재 날씨는 구름이 많고, 기온은 약 15도 정도이며, 바람은 북서풍 약간 강하게 불고 있습니다. 습도는 20~30% 정도입니다.

더 구체적인 시간별 날씨나 상세한 기상 정보가 필요하시면 알려주세요!


### OpenAI Tools Agent

In [27]:
from langchain_classic.agents import load_tools
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-4.1-mini",temperature=0)
tools=load_tools(["ddg-search"])
prompt = hub.pull("hwchase17/openai-tools-agent")

agent=create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

result=agent_executor.invoke({"input": "서울과 부산의 날씨를 알려줘"})



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': '서울 날씨'}`


Источник: Радиостанция " 서울 날씨 ". Сохранить. Автовоспроизведение.Реклама Ju Yoon Ha • 서울 날씨 • 2019. 서울 . Weather Warning Indicator. 서울 - 7일 기상 예보. You can add in NALSSHI ( 날씨 ) to be more noun-specific with your phrases. NALSSHI means “weather.” 서울 날씨 어때요? Seoul nalsshi eoddaeyo? How’s the weather in Seoul? TikTok에서 서울 날씨 치금 관련 동영상을 찾아보세요. 오늘# 서울 날씨 실화니?. 여름 안에서 by 싹쓰리 (Feat. 황광희) - SSAK3 (U-do-ragon & Linda G & B-ryong). 서울 올해 최고 기온이자 7월 상순 기준 역대 가장 더운 날씨를 기록했습니다.폭염경보가 내려진 가운데 서울 낮 동안 빠르게 기온이 오르더니 현재 시각 37.8도를 기록했습니다.
Invoking: `duckduckgo_search` with `{'query': '부산 날씨'}`


2025. 11. 10. · 내일 · 시각: 01시 · 날씨: 구름 많음 · 기온(체감온도) -4℃(-10℃) · 체감온도: -10℃ · 강수량: - · - · 강수확률: 20% · 바람: 북서풍 약간 강 4m/s ... 2025. 5. 9. · 부산지방기상청만의 날씨 서비스를 소개합니다ㅣ부산지방기상청 사무실&장비 공개! 11K views · 8 months ago. #기상청 #국민정책기자단 #부산지방기상청 ...more ... 2025. 12. 3. · 부산 시간별날씨 ; 04시 · 4.5m/s · 습도 20% ; 05시 · 5.3m/s · 습도 20% ; 06

In [28]:
print(result["output"])

현재 서울과 부산의 날씨 정보를 찾는 데 어려움이 있었습니다. 대신, 최신 날씨 정보를 확인하시려면 기상청 웹사이트나 날씨 앱을 이용하시는 것을 추천드립니다. 필요하시면 다시 시도해 드릴 수 있습니다.


### Function calling을 활용한 OutputParser, Extraction, Tagging

In [31]:
import json
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_extraction_chain

schema = {
    "properties":{
        "person_name":{"type":"string"},
        "person_age":{"type":"integer"},
        "person_hair_color":{"type":"string"},
        "dog_name":{"type":"string"},
        "dog_breed":{"type":"string"},
    },
    "required":["person_name", "person_height"],
}

text = """Alex is 5 feet tall, Claudia is 1 feet taller Alex and jumps higher than him. Claudia is a brunette and Alex is blonde.
Alex's dog Frosty is a labrador and likes to play hide and seek."""

chat = ChatOpenAI(model="gpt-4.1-mini",temperature=0)
chain = create_extraction_chain(schema, chat)
people = chain.invoke(text)


C:\Users\User\AppData\Local\Temp\ipykernel_9692\1218785918.py:20: LangChainDeprecationWarning: LangChain has introduced a method called `with_structured_output` thatis available on ChatModels capable of tool calling.You can read more about the method here: <https://docs.langchain.com/oss/python/langchain/models#structured-outputs>.
  chain = create_extraction_chain(schema, chat)


In [32]:
print(json.dumps(people,indent=2))

{
  "input": "Alex is 5 feet tall, Claudia is 1 feet taller Alex and jumps higher than him. Claudia is a brunette and Alex is blonde.\nAlex's dog Frosty is a labrador and likes to play hide and seek.",
  "text": [
    {
      "person_name": "Alex",
      "person_hair_color": "blonde"
    },
    {
      "person_name": "Claudia",
      "person_hair_color": "brunette"
    },
    {
      "person_name": "Frosty",
      "dog_breed": "labrador"
    }
  ]
}


### Evaluation

In [34]:
from langchain_openai import ChatOpenAI
from langchain_classic.evaluation import load_evaluator

chat = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
evaluator = load_evaluator("qa",eval_11m=chat)

result = evaluator.evaluate_strings(
    input="""나는 시장에 가서 사과 10개를 샀어.
사과 2개를 이웃에게 주고, 2개를 수리공에게 주었어.
그리고 사과 5개를 더 사서 1개는 내가 먹었어.
나는 몇 개의 사과를 가지고 있었니?""",
    prediction="""먼저 사과 10개로 시작했어.
이웃에게 2개, 수리공에게 2개를 나누어 주었으므로 사과가 6개가 남았어.
그런 다음 사과 5개를 더 사서 이제 사과가 11개가 되었어.
마지막으로 사과 1개를 먹었으므로 사과 10개가 남게 돼.""",
reference="10개",

)

In [35]:
print(result)

{'reasoning': 'CORRECT', 'value': 'CORRECT', 'score': 1}
